# 04 — Agent 2 : fusion multi-échelles et cohérence sémantique

Ce notebook combine :

```text
03    → annotations géométriques/topographiques du GPX
03bis → annotations terrain OSM
```

Le problème corrigé dans cette V2 est important :

```text
montee + descente
```

ne doit jamais apparaître sur une même portion finale.

De même :

```text
pente_douce + forte_pente
```

est interdit sur une même portion finale.

On impose donc deux contraintes :

```text
slope_direction :
    montee XOR descente

slope_intensity :
    pente_douce XOR forte_pente
```

`lacets` et `foret` restent indépendants.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

## 1. Charger les annotations du 03 et du 03bis

Le fichier forêt est optionnel pour permettre de continuer à tester
le pipeline sans connexion OSM.

In [ ]:
geometry_path = Path(
    "data/output/agent1_annotations_atomic.json"
)

forest_path = Path(
    "data/output/terrain_forest_annotations.json"
)

route_path = Path(
    "data/output/route_resampled.csv"
)

if not geometry_path.exists():
    raise FileNotFoundError(
        "Exécute d'abord le notebook 03."
    )

if not route_path.exists():
    raise FileNotFoundError(
        "Exécute d'abord le notebook 02."
    )

with geometry_path.open("r", encoding="utf-8") as f:
    geometry_annotations = json.load(f)

if forest_path.exists():
    with forest_path.open("r", encoding="utf-8") as f:
        terrain_annotations = json.load(f)

    print(
        "Annotations terrain chargées :",
        len(terrain_annotations)
    )
else:
    terrain_annotations = []

    print(
        "ATTENTION : pas de fichier forêt. "
        "Le notebook 04 continue sans terrain."
    )

route = pd.read_csv(route_path)

raw_annotations = (
    geometry_annotations
    + terrain_annotations
)

print(
    "Annotations géométriques :",
    len(geometry_annotations)
)
print(
    "Annotations totales :",
    len(raw_annotations)
)

## 2. Modèles de données

La portion finale possède maintenant une structure sémantique explicite.

Une seule valeur peut exister pour `slope_direction` et une seule pour
`slope_intensity`. Les contradictions deviennent donc structurellement
impossibles dans la sortie.

In [ ]:
class Annotation(BaseModel):
    feature: str
    family: str | None = None
    start_m: float
    end_m: float
    resolution_m: float
    confidence: float
    metrics: dict = Field(default_factory=dict)


class Portion(BaseModel):
    start_m: float
    end_m: float

    slope_direction: str | None = None
    slope_intensity: str | None = None

    geometry_features: list[str] = Field(
        default_factory=list
    )

    terrain_features: list[str] = Field(
        default_factory=list
    )

    # Vue plate conservée pour compatibilité avec l'Agent 3.
    annotations: list[str] = Field(
        default_factory=list
    )

    details: dict = Field(
        default_factory=dict
    )

In [ ]:
def infer_family(feature):
    if feature in {"montee", "descente"}:
        return "slope_direction"

    if feature in {
        "pente_douce",
        "forte_pente",
    }:
        return "slope_intensity"

    if feature == "lacets":
        return "geometry"

    if feature == "foret":
        return "terrain"

    return "other"


annotations = []

for raw in raw_annotations:
    raw = dict(raw)

    if not raw.get("family"):
        raw["family"] = infer_family(
            raw["feature"]
        )

    annotations.append(
        Annotation(**raw)
    )

## 3. Principe de consolidation multi-échelles

On ne fusionne plus aveuglément de grandes zones avant de résoudre les conflits.

À la place, on travaille d'abord sur les intervalles de base de la route :

```text
0–10
10–20
20–30
...
```

Pour chaque intervalle, on rassemble toutes les preuves qui le couvrent :

```text
montee 50 m
montee 100 m
descente 50 m
forte_pente 25 m
lacets 150 m
foret OSM
```

Puis on résout les groupes incompatibles.

In [ ]:
CONFLICT_MARGIN = 0.05

# Petit bonus si une caractéristique est soutenue
# par plusieurs résolutions différentes.
MULTISCALE_BONUS_PER_EXTRA_SCALE = 0.05


def overlap_length(a_start, a_end, b_start, b_end):
    return max(
        0.0,
        min(a_end, b_end)
        - max(a_start, b_start),
    )


def evidence_for_interval(
    annotations,
    start_m,
    end_m,
):
    active = []

    interval_length = max(
        end_m - start_m,
        1e-9,
    )

    for annotation in annotations:
        overlap = overlap_length(
            start_m,
            end_m,
            annotation.start_m,
            annotation.end_m,
        )

        if overlap <= 0:
            continue

        active.append({
            "annotation": annotation,
            "overlap_fraction": (
                overlap / interval_length
            ),
        })

    return active


def summarize_feature_evidence(
    active_evidence,
):
    by_feature = defaultdict(list)

    for item in active_evidence:
        annotation = item["annotation"]

        by_feature[
            annotation.feature
        ].append(item)

    summaries = {}

    for feature, items in by_feature.items():
        weighted_confidences = [
            (
                item["annotation"].confidence
                * item["overlap_fraction"]
            )
            for item in items
        ]

        base_score = float(
            np.mean(weighted_confidences)
        )

        resolutions = sorted({
            float(
                item["annotation"].resolution_m
            )
            for item in items
        })

        bonus = (
            MULTISCALE_BONUS_PER_EXTRA_SCALE
            * max(0, len(resolutions) - 1)
        )

        score = min(
            1.0,
            base_score + bonus,
        )

        summaries[feature] = {
            "score": score,
            "n_evidence": len(items),
            "resolutions_m": resolutions,
            "mean_confidence": float(
                np.mean([
                    item["annotation"].confidence
                    for item in items
                ])
            ),
        }

    return summaries

## 4. Pente locale comme arbitre en cas de conflit serré

Si `montee` et `descente` ont des scores presque identiques sur un intervalle,
on regarde le signe de la variation d'altitude locale.

Pour l'intensité, si `pente_douce` et `forte_pente` sont presque à égalité,
on utilise la valeur absolue de cette pente locale comme arbitre.

Le LLM n'intervient pas dans cette résolution : elle est déterministe.

In [ ]:
route_d = route["distance_m"].to_numpy(
    dtype=float
)

route_z = route["elev_m"].to_numpy(
    dtype=float
)


def local_slope_pct(start_m, end_m):
    if end_m <= start_m:
        return 0.0

    z0 = np.interp(
        start_m,
        route_d,
        route_z,
    )

    z1 = np.interp(
        end_m,
        route_d,
        route_z,
    )

    return float(
        100.0
        * (z1 - z0)
        / (end_m - start_m)
    )

## 5. Résoudre `montee XOR descente`

In [ ]:
def resolve_direction(
    feature_summary,
    start_m,
    end_m,
):
    up = feature_summary.get("montee")
    down = feature_summary.get("descente")

    if up is None and down is None:
        return None

    if up is not None and down is None:
        return "montee"

    if down is not None and up is None:
        return "descente"

    diff = up["score"] - down["score"]

    if abs(diff) > CONFLICT_MARGIN:
        return (
            "montee"
            if diff > 0
            else "descente"
        )

    # Conflit serré : arbitre physique local.
    slope = local_slope_pct(
        start_m,
        end_m,
    )

    if slope > 0:
        return "montee"

    if slope < 0:
        return "descente"

    # Cas parfaitement plat / indéterminé.
    return None

## 6. Résoudre `pente_douce XOR forte_pente`

In [ ]:
def resolve_intensity(
    feature_summary,
    start_m,
    end_m,
):
    gentle = feature_summary.get(
        "pente_douce"
    )

    strong = feature_summary.get(
        "forte_pente"
    )

    if gentle is None and strong is None:
        return None

    if gentle is not None and strong is None:
        return "pente_douce"

    if strong is not None and gentle is None:
        return "forte_pente"

    diff = (
        strong["score"]
        - gentle["score"]
    )

    if abs(diff) > CONFLICT_MARGIN:
        return (
            "forte_pente"
            if diff > 0
            else "pente_douce"
        )

    # Conflit serré :
    # l'amplitude locale sert d'arbitre.
    abs_slope = abs(
        local_slope_pct(
            start_m,
            end_m,
        )
    )

    return (
        "forte_pente"
        if abs_slope >= 10.0
        else "pente_douce"
    )

## 7. Construire un état sémantique par intervalle de base

In [ ]:
segment_states = []

distances = route["distance_m"].to_numpy(
    dtype=float
)

for start_m, end_m in zip(
    distances[:-1],
    distances[1:],
):
    active = evidence_for_interval(
        annotations,
        float(start_m),
        float(end_m),
    )

    summary = summarize_feature_evidence(
        active
    )

    direction = resolve_direction(
        summary,
        float(start_m),
        float(end_m),
    )

    intensity = resolve_intensity(
        summary,
        float(start_m),
        float(end_m),
    )

    geometry_features = []

    if "lacets" in summary:
        geometry_features.append("lacets")

    terrain_features = []

    if "foret" in summary:
        terrain_features.append("foret")

    flat_annotations = []

    if direction is not None:
        flat_annotations.append(direction)

    if intensity is not None:
        flat_annotations.append(intensity)

    flat_annotations.extend(
        geometry_features
    )

    flat_annotations.extend(
        terrain_features
    )

    segment_states.append({
        "start_m": float(start_m),
        "end_m": float(end_m),
        "slope_direction": direction,
        "slope_intensity": intensity,
        "geometry_features": geometry_features,
        "terrain_features": terrain_features,
        "annotations": flat_annotations,
        "local_slope_pct": local_slope_pct(
            float(start_m),
            float(end_m),
        ),
        "feature_summary": summary,
    })

segment_df = pd.DataFrame([
    {
        "start_m": s["start_m"],
        "end_m": s["end_m"],
        "slope_direction": s["slope_direction"],
        "slope_intensity": s["slope_intensity"],
        "geometry": ",".join(
            s["geometry_features"]
        ),
        "terrain": ",".join(
            s["terrain_features"]
        ),
        "annotations": ",".join(
            s["annotations"]
        ),
        "local_slope_pct": round(
            s["local_slope_pct"],
            3,
        ),
    }
    for s in segment_states
])

display(segment_df.head(30))

## 8. Vérification automatique des contradictions

Ces assertions doivent toujours passer.

In [ ]:
for state in segment_states:
    annotations_set = set(
        state["annotations"]
    )

    assert not (
        "montee" in annotations_set
        and "descente" in annotations_set
    ), state

    assert not (
        "pente_douce" in annotations_set
        and "forte_pente" in annotations_set
    ), state

print(
    "OK — aucune portion de base ne contient "
    "montee+descente ou pente_douce+forte_pente."
)

## 9. Fusionner les intervalles consécutifs de même état

Si 30 intervalles successifs de 10 m ont exactement le même état sémantique,
ils deviennent une seule portion de 300 m.

In [ ]:
def semantic_key(state):
    return (
        state["slope_direction"],
        state["slope_intensity"],
        tuple(state["geometry_features"]),
        tuple(state["terrain_features"]),
    )


portions = []

for state in segment_states:
    # Ne pas créer de portion vide.
    if not state["annotations"]:
        continue

    current = Portion(
        start_m=state["start_m"],
        end_m=state["end_m"],
        slope_direction=state[
            "slope_direction"
        ],
        slope_intensity=state[
            "slope_intensity"
        ],
        geometry_features=state[
            "geometry_features"
        ],
        terrain_features=state[
            "terrain_features"
        ],
        annotations=state[
            "annotations"
        ],
        details={
            "segments": [
                {
                    "start_m": state["start_m"],
                    "end_m": state["end_m"],
                    "local_slope_pct": (
                        state["local_slope_pct"]
                    ),
                    "feature_summary": (
                        state["feature_summary"]
                    ),
                }
            ]
        },
    )

    if not portions:
        portions.append(current)
        continue

    previous = portions[-1]

    previous_key = (
        previous.slope_direction,
        previous.slope_intensity,
        tuple(previous.geometry_features),
        tuple(previous.terrain_features),
    )

    current_key = semantic_key(state)

    if (
        abs(
            previous.end_m
            - current.start_m
        ) < 1e-6
        and previous_key == current_key
    ):
        previous.end_m = current.end_m

        previous.details[
            "segments"
        ].extend(
            current.details["segments"]
        )

    else:
        portions.append(current)

## 10. Résultat final de l'Agent 2

In [ ]:
portions_df = pd.DataFrame([
    {
        "start_m": p.start_m,
        "end_m": p.end_m,
        "length_m": p.end_m - p.start_m,
        "direction": p.slope_direction,
        "intensity": p.slope_intensity,
        "geometry": ", ".join(
            p.geometry_features
        ),
        "terrain": ", ".join(
            p.terrain_features
        ),
        "annotations": ", ".join(
            p.annotations
        ),
    }
    for p in portions
])

print(
    "Nombre de portions finales :",
    len(portions)
)

display(portions_df)

## 11. Contrôle global final

In [ ]:
contradictions = []

for p in portions:
    values = set(p.annotations)

    if (
        "montee" in values
        and "descente" in values
    ):
        contradictions.append(
            (p.start_m, p.end_m, "direction")
        )

    if (
        "pente_douce" in values
        and "forte_pente" in values
    ):
        contradictions.append(
            (p.start_m, p.end_m, "intensity")
        )

if contradictions:
    raise AssertionError(
        f"Contradictions restantes : "
        f"{contradictions}"
    )

print(
    "VALIDATION OK : aucune contradiction "
    "sémantique dans les portions finales."
)

## 12. Export pour l'Agent 3

In [ ]:
output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

segment_path = (
    output_dir / "agent2_segment_states.csv"
)

portions_json_path = (
    output_dir / "agent2_portions.json"
)

portions_csv_path = (
    output_dir / "agent2_portions.csv"
)

segment_df.to_csv(
    segment_path,
    index=False,
)

with portions_json_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        [p.model_dump() for p in portions],
        f,
        ensure_ascii=False,
        indent=2,
    )

portions_df.to_csv(
    portions_csv_path,
    index=False,
)

print("Exports Agent 2 :")
print("-", segment_path)
print("-", portions_json_path)
print("-", portions_csv_path)

## Architecture obtenue

```text
03 — GPX / Agent 1
├── montee
├── descente
├── pente_douce
├── forte_pente
└── lacets
         │
         │
03bis — OSM
└── foret
         │
         ▼
04 — Agent 2
├── preuves multi-échelles
├── résolution montee XOR descente
├── résolution douce XOR forte
├── combinaison lacets
├── combinaison foret
└── fusion des portions successives
```